# 2. GraphRAG: Retrieval-Augmented Generation over the Knowledge Graph

**Part of the AlzRAGBench notebook series** (`Diffusion_AlzRAGBench_v2.0/`).

**Requires a GPU with roughly >=8GB VRAM.** This notebook loads a 3B-parameter
**diffusion language model** in plain bf16 for answer generation. It will not run
usefully on a CPU-only machine -- run it on a cloud/rented GPU instance (or a decent
local GPU).

## What is GraphRAG, and why a diffusion LLM?

GraphRAG answers a question by retrieving a **relevant subgraph** of entities and
relations from a knowledge graph (rather than retrieving raw text chunks, as VectorRAG
does in Notebook 3), formatting that subgraph as textual context, and asking a language
model to answer the question grounded in that context. It excels at questions that
require **connecting multiple facts** that may never appear together in a single
document chunk.

For generation, we use **SDLM-3B-D4**, a genuine **diffusion language model** --
fundamentally different from a standard autoregressive LLM (GPT-style, left-to-right,
one token at a time with causal attention), but *semi-autoregressive* rather than
fully parallel:

- Generation still starts from masked-out positions and denoises them, but only
  `n_future_tokens` (here, 4 -- the "D4" in the model name) positions at a time, in a
  block.
- Unlike a fully-parallel diffusion model, SDLM keeps a **KV cache** across blocks the
  way a normal autoregressive model does, reusing previously-computed
  keys/values instead of reprocessing the whole sequence every step.
- Within each block, the model predicts all masked positions at once and accepts the
  longest run of confident predictions (starting from the first); anything past the
  first low-confidence position is discarded and retried next step.

This "sequential diffusion" design (fine-tuned from Qwen2.5-3B) is what lets SDLM be
both small (3B, no quantization needed) and fast (KV-cache reuse), while still being a
genuine diffusion LM rather than a standard autoregressive model. See Section 2.9-2.10
for the full explanation and the sampling algorithm implementation.

## What this notebook builds

1. Rebuild the knowledge graph (same as Notebook 1, self-contained -- no dependency on
   its output).
2. A simple **entity linking** step: map free-text question mentions to graph node IDs.
3. **Four retrieval strategies**, each suited to a different kind of question:
   - **1-hop neighbor retrieval** -- direct facts about a single entity.
   - **Hub-avoiding multi-hop BFS** -- chains of facts connecting entities that are 2+
     hops apart, without exploding into "everything connects to everything" on our
     small, densely-connected demo graph.
   - **Shortest-path retrieval** -- the specific reasoning chain connecting two
     mentioned entities.
   - **Centrality-guided global context** -- a "big picture" summary from the graph's
     most important hub entities, for broad synthesis questions.
4. Load SDLM-3B-D4 and implement its block-wise semi-autoregressive sampling
   algorithm.
5. Showcase all four retrieval strategies end-to-end on real questions from
   `../Dataset/Evaluation/eval_dataset.json`.

## 2.1 Install dependencies

Run once per environment. SDLM-3B-D4 is small enough to run in plain bf16 with no
quantization library needed, so `bitsandbytes` has been dropped from this version's
dependencies. `transformers` is pinned to `4.37.2` deliberately -- see Section 2.9 for
why.

In [ ]:
# !pip install -q torch "transformers==4.37.2" accelerate pandas networkx

## 2.2 Imports and GPU check

We print the detected GPU name and total VRAM up front -- if this shows no GPU (or a
GPU with well under 8GB), stop here and switch to a GPU runtime before continuing,
since Section 2.9 will try to load SDLM-3B-D4. We also set
`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` here, before any CUDA call happens
-- part of the fix (see Section 2.10) for the "VRAM creeps up with every generation and
eventually OOMs" problem from the previous (LLaDA) version of this notebook.

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import re
from pathlib import Path

import networkx as nx
import pandas as pd
import torch

DATASET_DIR = Path("Dataset")
KG_DIR = DATASET_DIR / "Knowledge graph"
EVAL_PATH = DATASET_DIR / "Evaluation" / "eval_dataset.json"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no CUDA GPU detected. Section 2.9+ (loading SDLM-3B-D4) requires "
          "a GPU with roughly >=8GB VRAM and will fail or be extremely slow on CPU. "
          "Retrieval-only sections (2.3-2.6) will still run fine.")

GPU: Tesla T4
VRAM: 15.6 GB


## 2.3 Rebuild the knowledge graph

Identical construction to Notebook 1 (loaded fresh from the CSVs, not from any saved
graph object), so this notebook can run entirely on its own.


In [2]:
nodes_df = pd.read_csv(KG_DIR / "nodes.csv")
edges_df = pd.read_csv(KG_DIR / "edges.csv")

G = nx.MultiDiGraph()
for _, row in nodes_df.iterrows():
    G.add_node(row["nodeId:ID"], name=row["name"], type=row["type:LABEL"], description=row["description"])
for _, row in edges_df.iterrows():
    G.add_edge(
        row[":START_ID"], row[":END_ID"],
        relation=row[":TYPE"], source_articles=row["source_articles"], evidence=row["evidence"],
    )

degree_centrality = nx.degree_centrality(G.to_undirected())

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Graph: 54 nodes, 89 edges


## 2.4 Entity linking: from question text to graph node IDs

Before we can retrieve *anything* from the graph, we need to figure out which node(s)
a question is talking about. Production GraphRAG systems typically use a trained NER /
entity-linking model (e.g. scispaCy + UMLS linking for biomedical text). For a
54-node graph, that's overkill -- instead we build a simple **alias index**:

- Every node's own ID and full name become aliases (lowercased).
- CamelCase node IDs are split into readable phrases (e.g. `AmyloidBeta` -> "amyloid beta").
- Parenthetical qualifiers are stripped as an extra alias (e.g. "Amyloid-beta (Abeta)" -> "amyloid-beta").
- A small hand-curated dictionary adds domain abbreviations and phrasing a real
  question is likely to use (`MCI`, `ARIA`, `APOE4`, "poor sleep", etc.) that
  wouldn't be derivable automatically from the node name alone.

Matching uses **word-boundary regex matching** (not naive substring search, which would
wrongly match "AD" inside a word like "adopted"), and prefers longer/more specific
aliases first so "vascular dementia" is matched as one entity rather than accidentally
also matching the substring "dementia" as a second, less specific one.

**Known limitation** (worth stating plainly, not hiding): this is substring/alias
matching, not true entity recognition -- it will miss entities mentioned only
implicitly or via phrasing we didn't anticipate (e.g. a question about "sex" alone,
without the phrase "sex differences," won't link to the `SexDifferences` node). We'll
see this play out in the examples below, and it's a genuine, honest limitation of
lightweight GraphRAG entity linking, not a bug to paper over.


In [3]:
MANUAL_ALIASES = {
    "AD": ["ad", "alzheimer's disease", "alzheimer disease", "alzheimers"],
    "MCI": ["mci", "mild cognitive impairment"],
    "APOE_e4": ["apoe4", "apoe e4", "apoe epsilon-4", "epsilon-4", "e4 allele"],
    "APOE_e2": ["apoe2", "apoe e2", "apoe epsilon-2", "epsilon-2", "e2 allele"],
    "AmyloidBeta": ["amyloid-beta", "amyloid beta", "abeta", "a-beta"],
    "NFT": ["neurofibrillary tangles", "tangles", "nft"],
    "ARIA": ["aria", "amyloid-related imaging abnormalities"],
    "APP": ["app gene", "amyloid precursor protein", "app"],
    "DownSyndrome": ["down syndrome", "trisomy 21"],
    "AntiAmyloidAntibodies": ["anti-amyloid", "monoclonal antibodies", "anti-amyloid antibodies"],
    "CholinesteraseInhibitors": ["cholinesterase inhibitor", "cholinesterase inhibitors", "chei"],
    "GlymphaticSystem": ["glymphatic", "glymphatic system"],
    "NMDAReceptor": ["nmda receptor", "nmda"],
    "FluidBiomarkers": ["csf biomarkers", "plasma biomarkers", "abeta42/40"],
    "MRIAtrophy": ["mri", "structural mri", "brain atrophy"],
    "AmyloidPET": ["amyloid pet"],
    "TauPET": ["tau pet"],
    "CardiovascularRisk": ["vascular risk factors", "cardiovascular risk factors", "vascular risk"],
    "VascularDementia": ["vascular dementia"],
    "MixedDementia": ["mixed dementia"],
    "SexDifferences": ["sex differences", "gender differences", "sex and gender"],
    "AmyloidCascadeHypothesis": ["amyloid cascade hypothesis", "amyloid hypothesis"],
    "SleepDisturbance": ["poor sleep", "sleep disturbance", "sleep"],
    "Aging": ["aging", "ageing"],
}


def build_alias_index(G):
    alias_to_node = {}

    def register(alias, node_id):
        alias = alias.lower().strip()
        if len(alias) < 3:
            return
        alias_to_node.setdefault(alias, node_id)

    for node_id, data in G.nodes(data=True):
        register(node_id, node_id)
        register(data["name"], node_id)
        spaced = re.sub(r"(?<!^)(?=[A-Z])", " ", node_id).lower()
        register(spaced, node_id)
        no_parens = re.sub(r"\s*\([^)]*\)", "", data["name"])
        register(no_parens, node_id)

    for node_id, aliases in MANUAL_ALIASES.items():
        for a in aliases:
            register(a, node_id)

    return alias_to_node


ALIAS_INDEX = build_alias_index(G)
SORTED_ALIASES = sorted(ALIAS_INDEX.keys(), key=len, reverse=True)  # longest/most specific first


def link_entities(question, max_entities=6):
    q = question.lower()
    found = []
    covered_spans = []

    def overlaps(start, end):
        return any(not (end <= s or start >= e) for s, e in covered_spans)

    for alias in SORTED_ALIASES:
        pattern = r"(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])"
        for m in re.finditer(pattern, q):
            if overlaps(m.start(), m.end()):
                continue
            node_id = ALIAS_INDEX[alias]
            if node_id not in found:
                found.append(node_id)
            covered_spans.append((m.start(), m.end()))
    return found[:max_entities]


def filter_dominant_hubs(entities, degree_centrality, threshold=0.3):
    """Almost every question in this corpus mentions 'Alzheimer's disease' somewhere,
    which links the AD mega-hub node (0.62 degree centrality, ~5x the next highest)
    alongside whatever specific entity the question is actually about. Left alone,
    AD's ~30 edges bury the specific entity's handful of relevant edges once we
    truncate the printed/prompted context. So: if we found at least one non-hub
    entity, drop the hub entity/entities and keep only the specific one(s) -- the
    hub's own facts about those specific entities still surface via the specific
    entity's own edges anyway. Only fall back to the hub if it's *all* we found."""
    non_hub = [e for e in entities if degree_centrality.get(e, 0) <= threshold]
    return non_hub if non_hub else entities


print(f"Built alias index with {len(ALIAS_INDEX)} aliases covering {G.number_of_nodes()} nodes.")


Built alias index with 164 aliases covering 54 nodes.


In [4]:
# Quick sanity check on a few example questions
for q in [
    "What is memantine's mechanism of action in treating Alzheimer's disease?",
    "Why is Down syndrome considered a natural model supporting the amyloid cascade hypothesis?",
    "How does carrying the APOE e4 allele interact with sex to affect Alzheimer's risk?",
]:
    print(f"Q: {q}")
    print(f"  Linked entities: {link_entities(q)}\n")


Q: What is memantine's mechanism of action in treating Alzheimer's disease?
  Linked entities: ['AD', 'Memantine']

Q: Why is Down syndrome considered a natural model supporting the amyloid cascade hypothesis?
  Linked entities: ['AmyloidCascadeHypothesis', 'DownSyndrome']

Q: How does carrying the APOE e4 allele interact with sex to affect Alzheimer's risk?
  Linked entities: ['APOE_e4', 'APOE']



## 2.5 Retrieval Method 1 -- Direct neighbor (1-hop) retrieval

The simplest strategy: for each linked entity, fetch every edge directly touching it
(both outgoing and incoming) and present those as context. This is ideal for questions
that are really about **one entity's direct properties or relationships** -- e.g. "what
is memantine's mechanism of action?" doesn't need multi-hop reasoning, just Memantine's
immediate edges.


In [5]:
def edge_record(u, v, data):
    return {
        "start": u, "end": v,
        "start_name": G.nodes[u]["name"], "end_name": G.nodes[v]["name"],
        "relation": data["relation"], "evidence": data["evidence"],
        "source_articles": data["source_articles"],
    }


def dedupe_edges(edges):
    seen, out = set(), []
    for e in edges:
        key = (e["start"], e["end"], e["relation"])
        if key not in seen:
            seen.add(key)
            out.append(e)
    return out


def retrieve_1hop(entities, G):
    results = []
    for e in entities:
        if e not in G:
            continue
        for _, v, data in G.out_edges(e, data=True):
            results.append(edge_record(e, v, data))
        for u, _, data in G.in_edges(e, data=True):
            results.append(edge_record(u, e, data))
    return dedupe_edges(results)


def format_context(edges, limit=15):
    lines = [
        f"- {e['start_name']} --[{e['relation']}]--> {e['end_name']}: {e['evidence']} (source: {e['source_articles']})"
        for e in edges[:limit]
    ]
    return "\n".join(lines)


In [6]:
demo_q1 = "What is memantine's mechanism of action in treating Alzheimer's disease?"
# Every question in this corpus mentions "Alzheimer's disease," so raw linking also
# returns the AD hub node alongside Memantine -- filter_dominant_hubs (defined above)
# drops it so the printed/prompted context stays focused on what the question is
# actually about, instead of AD's ~30 generic edges burying Memantine's few relevant ones.
entities = filter_dominant_hubs(link_entities(demo_q1), degree_centrality)
print(f"Q: {demo_q1}")
print(f"Linked entities: {entities}\n")

edges_1hop = retrieve_1hop(entities, G)
print(f"Retrieved {len(edges_1hop)} 1-hop edges:\n")
print(format_context(edges_1hop))


Q: What is memantine's mechanism of action in treating Alzheimer's disease?
Linked entities: ['Memantine']

Retrieved 2 1-hop edges:

- Memantine --[ANTAGONIST_OF]--> NMDA Receptor: Memantine is an uncompetitive NMDA receptor antagonist. (source: pubmed_12672860;wikipedia_memantine)
- Memantine --[TREATS]--> Alzheimer's Disease: A placebo-controlled trial found memantine reduced clinical deterioration in moderate-to-severe AD. (source: pubmed_12672860)


## 2.6 Retrieval Method 2 -- Hub-avoiding multi-hop BFS

For questions that require **chaining facts through an intermediate concept** (e.g.
"how does poor sleep contribute to protein aggregation?" -- sleep affects the
glymphatic system, which clears amyloid-beta), 1-hop retrieval isn't enough.

A naive fix is breadth-first search out to *k* hops. But on our small, densely
connected 54-node graph, unrestricted 2-hop BFS from almost any starting entity ends up
touching 35+ nodes (most of the graph) -- because everything eventually routes through
the `AD` mega-hub node (degree centrality 0.62, vs. ~0.1 for the next most-connected
nodes). That defeats the purpose of a *targeted* retrieval strategy.

The fix used here: **treat high-degree hub nodes as stop points**. We still record the
direct edges touching a hub, but we don't expand *further* outward through it. This
keeps multi-hop retrieval meaningfully different from "dump the whole graph" while
still finding real multi-hop chains through non-hub intermediate concepts (like
`SleepDisturbance -> GlymphaticSystem`).


In [7]:
def retrieve_multihop(entities, G, degree_centrality, max_hops=2, hub_threshold=0.3):
    visited = set(e for e in entities if e in G)
    frontier = set(visited)
    all_edges = []
    seen = set()
    for _ in range(max_hops):
        next_frontier = set()
        for node in frontier:
            is_hub = degree_centrality.get(node, 0) > hub_threshold
            for _, v, data in G.out_edges(node, data=True):
                key = (node, v, data["relation"])
                if key not in seen:
                    seen.add(key)
                    all_edges.append(edge_record(node, v, data))
                if v not in visited and not is_hub:
                    next_frontier.add(v)
            for u, _, data in G.in_edges(node, data=True):
                key = (u, node, data["relation"])
                if key not in seen:
                    seen.add(key)
                    all_edges.append(edge_record(u, node, data))
                if u not in visited and not is_hub:
                    next_frontier.add(u)
        visited |= next_frontier
        frontier = next_frontier
        if not frontier:
            break
    return all_edges, visited


In [8]:
demo_q2 = "Trace the chain of events by which poor sleep is thought to contribute to dementia-related protein aggregation."
entities = filter_dominant_hubs(link_entities(demo_q2), degree_centrality)
print(f"Q: {demo_q2}")
print(f"Linked entities: {entities}\n")

edges_multihop, visited_nodes = retrieve_multihop(entities, G, degree_centrality, max_hops=2)
print(f"Retrieved {len(edges_multihop)} edges across {len(visited_nodes)} visited nodes "
      f"(vs. {G.number_of_nodes()} total -- hub-avoidance kept this targeted):\n")
print(format_context(edges_multihop))


Q: Trace the chain of events by which poor sleep is thought to contribute to dementia-related protein aggregation.
Linked entities: ['SleepDisturbance', 'Dementia']

Retrieved 42 edges across 11 visited nodes (vs. 54 total -- hub-avoidance kept this targeted):

- Sleep Disturbance --[PRECEDES_ONSET_OF]--> Dementia: Disrupted sleep architecture frequently precedes clinical dementia onset. (source: pubmed_33004510)
- Physical Activity --[PROTECTIVE_FACTOR_FOR]--> Dementia: Physical activity is also associated with lower all-cause and vascular dementia incidence. (source: pubmed_35301183)
- Alzheimer's Disease --[SUBTYPE_OF]--> Dementia: AD accounts for roughly 60-70% of all dementia cases worldwide. (source: wikipedia_dementia)
- Vascular Dementia --[SUBTYPE_OF]--> Dementia: Vascular dementia is the second most common dementia subtype. (source: wikipedia_dementia)
- Sleep Disturbance --[IMPAIRS]--> Glymphatic System: Poor sleep reduces the glymphatic system's brain-clearance activity, wh

## 2.7 Retrieval Method 3 -- Shortest-path retrieval

When a question explicitly mentions **two (or more) entities** and asks how they
relate, the most direct GraphRAG strategy is to find the shortest path between them in
the graph and present the edges along that path as the "reasoning chain." This is the
clearest, most interpretable form of multi-hop retrieval -- the retrieved context *is*
the reasoning path.


In [9]:
def retrieve_shortest_path(entity_a, entity_b, G):
    undirected = G.to_undirected()
    try:
        path = nx.shortest_path(undirected, entity_a, entity_b)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None
    edges = []
    for u, v in zip(path[:-1], path[1:]):
        if G.has_edge(u, v):
            data = next(iter(G.get_edge_data(u, v).values()))
            edges.append(edge_record(u, v, data))
        elif G.has_edge(v, u):
            data = next(iter(G.get_edge_data(v, u).values()))
            edges.append(edge_record(v, u, data))
    return {"path": path, "edges": edges}


In [10]:
demo_q3 = "Why is Down syndrome considered a natural model supporting the amyloid cascade hypothesis, tracing the full genetic-to-molecular chain?"
print(f"Q: {demo_q3}\n")

# This question's full reasoning chain runs through the specific gene/chromosome
# mechanism rather than the more abstract "AmyloidCascadeHypothesis" node, so we
# target the path between DownSyndrome and AmyloidBeta directly.
sp = retrieve_shortest_path("DownSyndrome", "AmyloidBeta", G)
print(f"Shortest path: {' -> '.join(sp['path'])}\n")
print(format_context(sp["edges"]))


Q: Why is Down syndrome considered a natural model supporting the amyloid cascade hypothesis, tracing the full genetic-to-molecular chain?

Shortest path: DownSyndrome -> Chr21 -> APP -> AmyloidBeta

- Down Syndrome --[EXTRA_COPY_OF]--> Chromosome 21: Trisomy 21 gives an extra copy of chromosome 21 and therefore an extra APP gene copy. (source: textbook_statpearls_alzheimer_disease)
- Amyloid Precursor Protein / APP gene --[LOCATED_ON]--> Chromosome 21: The APP gene resides on chromosome 21. (source: textbook_statpearls_alzheimer_disease)
- Amyloid Precursor Protein / APP gene --[CLEAVED_TO_PRODUCE]--> Amyloid-beta (Abeta): Proteolytic cleavage of APP produces the amyloid-beta fragment central to AD pathology. (source: pubmed_31753135)


## 2.8 Retrieval Method 4 -- Centrality-guided global context

Broad synthesis questions (e.g. "what would a comprehensive prevention strategy look
like?") don't hinge on one or two specific entities -- they need **background context
from the graph's most structurally important concepts**. This mirrors the "global
search" mode in community-based GraphRAG systems (e.g. Microsoft's GraphRAG, which
summarizes graph communities for broad questions), simplified here to: take the top-k
nodes by degree centrality, and surface the edges connecting them to each other.


In [11]:
def retrieve_global_context(G, degree_centrality, top_k=10):
    top_nodes = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:top_k]
    top_ids = {n for n, _ in top_nodes}
    edges = []
    seen = set()
    for u, v, data in G.edges(data=True):
        if u in top_ids and v in top_ids:
            key = (u, v, data["relation"])
            if key not in seen:
                seen.add(key)
                edges.append(edge_record(u, v, data))
    return top_nodes, edges


In [12]:
top_nodes, global_edges = retrieve_global_context(G, degree_centrality, top_k=10)

print("Top-10 hub entities (by degree centrality):")
for node_id, score in top_nodes:
    print(f"  {node_id:25s} {score:.3f}   {G.nodes[node_id]['name']}")

print(f"\nEdges among the top-10 hubs ({len(global_edges)}):\n")
print(format_context(global_edges))


Top-10 hub entities (by degree centrality):
  AD                        0.623   Alzheimer's Disease
  AmyloidBeta               0.132   Amyloid-beta (Abeta)
  CholinesteraseInhibitors  0.113   Cholinesterase Inhibitors
  VascularDementia          0.094   Vascular Dementia
  MCI                       0.094   Mild Cognitive Impairment
  TauProtein                0.094   Tau Protein
  APOE                      0.094   Apolipoprotein E (APOE)
  AntiAmyloidAntibodies     0.094   Anti-Amyloid Monoclonal Antibodies
  Dementia                  0.075   Dementia
  AmyloidPlaques            0.075   Amyloid Plaques

Edges among the top-10 hubs (10):

- Alzheimer's Disease --[SUBTYPE_OF]--> Dementia: AD accounts for roughly 60-70% of all dementia cases worldwide. (source: wikipedia_dementia)
- Vascular Dementia --[COMORBID_WITH]--> Alzheimer's Disease: AD is frequently diagnosed alongside vascular dementia as mixed dementia. (source: wikipedia_dementia)
- Vascular Dementia --[SUBTYPE_OF]--> Dementi

## 2.9 Load SDLM-3B-D4

We now load the diffusion LLM that will generate answers from whichever retrieval
method's context we feed it. SDLM-3B-D4 ("Sequential Diffusion Language Model", block
size 4) is a 3B-parameter model fine-tuned from Qwen2.5-3B by OpenGVLab -- small enough
to run in plain bf16 (~6GB) with no quantization needed, comfortably fitting on a much
smaller GPU than the 16GB+ an 8B model would require.

Unlike a fully-parallel diffusion model (which re-processes the *entire* sequence with
bidirectional attention at every single denoising step and cannot use a KV cache),
SDLM is **semi-autoregressive**: it denoises `block_size` tokens at a time but keeps a
causal KV cache across blocks, reusing it the way a normal autoregressive model would.
This is both faster and dramatically lower-memory than the previous version of this
notebook's approach with LLaDA-8B.

`trust_remote_code=True` is required because SDLM uses a custom
`SDLMQwen2ForCausalLM` architecture that isn't part of the core `transformers`
library. `attn_implementation="eager"` is required too -- the model's custom code was
written and validated against the eager attention path, not `sdpa`.

**Pinned `transformers==4.37.2`: this is deliberate, not an oversight.** SDLM's custom
modeling code (`modeling_sdlm.py`) imports private helpers
(`_prepare_4d_causal_attention_mask_for_sdpa` from `transformers.modeling_attn_mask_utils`)
that later `transformers` releases removed when they reworked attention-masking
internals -- the exact kind of breakage that made the previous (LLaDA) version of this
notebook stop working on a fresh install. If you see an `ImportError`/`AttributeError`
mentioning `modeling_attn_mask_utils` or `_prepare_4d_causal_attention_mask`, that's
this incompatibility resurfacing on a `transformers` version bump; reinstall the pinned
version from Section 2.1, or check https://huggingface.co/OpenGVLab/SDLM-3B-D4 for a
current version note.

**One more thing you may hit: a `flash_attn` `ImportError` even with `attn_implementation="eager"`.**
`modeling_sdlm.py` guards its flash_attn import behind `if is_flash_attn_2_available():`,
but the `transformers` static import-scanner used for `trust_remote_code=True` loading
only understands `try/except`-guarded imports, not `if`-guarded ones -- so it flags
`flash_attn` as a hard requirement anyway, even though eager attention never touches
that code path. This is a known `transformers` limitation
(github.com/huggingface/transformers/issues/28459), not something specific to SDLM.
Rather than building `flash_attn` from source (painful, especially on Windows, and
unnecessary here), the loading cell below monkey-patches the scanner to ignore
`flash_attn` for this one file.

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch as _mock_patch

SDLM_MODEL_ID = "OpenGVLab/SDLM-3B-D4"


def _skip_flash_attn_check(filename):
    """See the flash_attn note above: modeling_sdlm.py's flash_attn import is guarded
    by `if is_flash_attn_2_available():`, but transformers' import scanner doesn't
    understand if-guards (only try/except) and flags flash_attn as required anyway,
    even though attn_implementation="eager" never touches that code path
    (github.com/huggingface/transformers/issues/28459)."""
    imports = get_imports(filename)
    if str(filename).endswith("modeling_sdlm.py") and "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports


with _mock_patch("transformers.dynamic_module_utils.get_imports", _skip_flash_attn_check):
    tokenizer = AutoTokenizer.from_pretrained(SDLM_MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        SDLM_MODEL_ID,
        trust_remote_code=True,
        attn_implementation="eager",
        torch_dtype=torch.bfloat16,
    )
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

print(f"Loaded {SDLM_MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


attn_mask_utils.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/SDLM-3B-D4:
- attn_mask_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/SDLM-3B-D4:
- attn_mask_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.84G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

Loaded OpenGVLab/SDLM-3B-D4
GPU memory allocated: 6.77 GB


## 2.10 The diffusion sampling algorithm (and the VRAM-growth fix)

SDLM is **semi-autoregressive**: rather than re-processing the whole sequence with
bidirectional attention at every step (as LLaDA did), it denoises `n_future_tokens`
positions at a time while keeping a standard causal **KV cache** across steps, the way
an autoregressive model does. At each step:

1. Append `n_future_tokens` placeholder `[MASK]` positions after the sequence
   generated so far.
2. Run one forward pass, reusing the cached keys/values from previous steps (so only
   the new positions actually need computing).
3. The model predicts a token for every masked position at once, with a confidence
   score for each.
4. Accept the longest **confident prefix** of those predictions (positions later in
   the block were predicted assuming the earlier ones land correctly, so once
   confidence drops below `threshold`, everything after that point in the block is
   discarded and re-predicted next step).
5. Roll the KV cache back to end exactly at the last accepted token, and repeat until
   `max_gen_len` tokens are generated or an end-of-sequence token is produced.

This is why SDLM needs far less VRAM per step than LLaDA: the forward pass only ever
processes `n_future_tokens` new positions against a cache, not the entire sequence
every time.

**The VRAM-growth/OOM bug from the previous (LLaDA) version of this notebook** came
from two things that compound across repeated calls: (1) never returning freed GPU
memory to the allocator's pool between generations, and (2) every question having a
different prompt/context length, which fragments that pool further since a
differently-sized old block can't be reused for a new request. The fix here has three
parts: `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (Section 2.2, reduces
fragmentation from variable-length sequences at the allocator level), explicitly
`del`-ing large intermediate tensors each denoising step instead of waiting for them to
fall out of scope, and calling `gc.collect()` + `torch.cuda.empty_cache()` after every
single `generate_with_sdlm()` call so freed memory is actually returned before the next
question starts.

In [14]:
import gc

import torch
import torch.nn.functional as F


def _top_p_logits(logits, top_p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    mask = torch.zeros_like(logits, dtype=torch.bool).scatter_(-1, sorted_indices, sorted_indices_to_remove)
    return logits.masked_fill(mask, torch.finfo(logits.dtype).min)


def _sample_tokens(logits, temperature=0.0, top_p=None, entropy_conf=False):
    """Returns (confidence, token_id) for each of the n_future_tokens candidate
    positions. Confidence is either top-probability (used to decide how many of the
    candidate tokens to "accept" this step) or a normalized-entropy score."""
    if temperature > 0:
        logits = logits / temperature
    if top_p is not None and top_p < 1:
        logits = _top_p_logits(logits, top_p)
    probs = torch.softmax(logits, dim=-1)

    if temperature > 0:
        x0 = torch.distributions.Categorical(probs=probs).sample()
        confidence = torch.gather(probs, -1, x0.unsqueeze(-1)).squeeze(-1)
    else:
        confidence, x0 = probs.max(dim=-1)

    if entropy_conf:
        entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
        max_entropy = torch.log(torch.tensor(float(probs.shape[-1]), device=entropy.device))
        confidence = 1.0 - (entropy / max_entropy)

    return confidence, x0


def _find_accept_length(confidence, threshold):
    """How many of the n_future_tokens candidate tokens (in order) are confident
    enough to commit this step -- SDLM accepts the longest confident *prefix*, since
    later candidate positions were predicted assuming all earlier ones land correctly."""
    mask = confidence > threshold
    first_low_confidence = torch.argmax((~mask).int(), dim=1)
    all_confident = mask.all(dim=1)
    seq_len = confidence.shape[1]
    return torch.where(all_confident, torch.full_like(first_low_confidence, seq_len), first_low_confidence)


@torch.no_grad()
def sdlm_generate(model, tokenizer, input_ids, max_gen_len=192, threshold=0.5,
                   n_future_tokens=4, temperature=0.0, top_p=None, alg="prob_conf"):
    """Semi-autoregressive block-diffusion generation with KV-cache reuse. Adapted from
    OpenGVLab's official SDLM_generate (github.com/OpenGVLab/SDLM), trimmed to the
    prob_conf/entropy_conf paths, with explicit intermediate-tensor cleanup added each
    iteration (see Section 2.10 markdown for why that matters).

    At each step: append n_future_tokens mask-token placeholders after the current
    sequence, run one forward pass reusing the cached keys/values from previous steps,
    and predict all n_future_tokens positions at once. Accept however many of those (in
    order, starting from the first) the model is confident enough about, roll the KV
    cache back to just past the accepted tokens, and repeat.
    """
    device = model.device
    mask_token_id = getattr(model.config, "text_mask_token_id", 151665)
    eos_ids = set(tokenizer.eos_token_id) if isinstance(tokenizer.eos_token_id, (list, tuple)) else {tokenizer.eos_token_id}
    gen_eos = getattr(model.generation_config, "eos_token_id", None)
    if gen_eos is not None:
        eos_ids |= set(gen_eos) if isinstance(gen_eos, (list, tuple)) else {gen_eos}

    model.model.block_size = getattr(model.config, "block_size", n_future_tokens)
    model.model.causal_attn = getattr(model.config, "causal_attn", False)
    model.model.text_mask_token_id = mask_token_id
    model.use_cache = True

    generated = input_ids.clone()
    prompt_len = generated.shape[1]
    target_len = prompt_len + max_gen_len
    past_key_values = None

    while generated.shape[1] < target_len:
        mask_tokens = torch.full((1, n_future_tokens - 1), mask_token_id, dtype=generated.dtype, device=device)
        generated_with_mask = torch.cat([generated, generated[:, -1:], mask_tokens], dim=1)

        start_idx = past_key_values[0][0].shape[2] if past_key_values is not None else 0
        position_ids = torch.arange(start_idx, generated_with_mask.shape[1], device=device).unsqueeze(0)
        position_ids[0, -n_future_tokens:] -= 1

        model_inputs = model.prepare_inputs_for_generation(
            generated_with_mask, past_key_values, None, use_cache=True, position_ids=position_ids,
        )
        outputs = model(**model_inputs)

        logits = outputs.logits[:, -n_future_tokens:, :]
        confidence, x0 = _sample_tokens(logits, temperature=temperature, top_p=top_p, entropy_conf=(alg == "entropy_conf"))
        accept_len = max(1, min(int(_find_accept_length(confidence, threshold)[0].item()), n_future_tokens))

        new_tokens = x0[0, :accept_len]
        eos_pos = next((i for i, t in enumerate(new_tokens.tolist()) if t in eos_ids), None)
        if eos_pos is not None:
            new_tokens = new_tokens[:eos_pos]

        generated = torch.cat([generated, new_tokens.unsqueeze(0)], dim=1)

        # Roll the KV cache back to only the tokens actually committed this step --
        # this, plus the cleanup below, is what keeps VRAM flat across many calls
        # instead of creeping upward the way the previous (LLaDA) notebook's
        # full-resequence-every-step approach did.
        past_key_values = tuple(
            (k[:, :, : generated.shape[1], :], v[:, :, : generated.shape[1], :])
            for k, v in outputs.past_key_values
        )
        del outputs, logits, confidence, x0, generated_with_mask, model_inputs

        if eos_pos is not None:
            break

    return generated[:, prompt_len:]


def generate_with_sdlm(prompt_text, system_prompt=None, max_gen_len=192, threshold=0.5,
                        n_future_tokens=4, temperature=0.0):
    """High-level helper: apply the chat template, run sdlm_generate, decode the
    response, and release GPU memory before returning -- see Section 2.10."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt_text})

    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_ids = tokenizer([formatted], return_tensors="pt").input_ids.to(model.device)

    response_ids = sdlm_generate(
        model, tokenizer, input_ids,
        max_gen_len=max_gen_len, threshold=threshold, n_future_tokens=n_future_tokens, temperature=temperature,
    )
    answer = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()

    del input_ids, response_ids
    gc.collect()
    torch.cuda.empty_cache()

    return answer

### A note on generation speed and parameters

Generation cost here scales with how many denoising *steps* actually run, which
depends on how often the model is confident enough to accept multiple tokens per step
-- a higher `threshold` means fewer tokens accepted per step (more steps, slower, more
conservative) and a lower `threshold` means more tokens accepted per step (fewer steps,
faster, more risk of a wrong token slipping through). `n_future_tokens=4` matches this
checkpoint's trained block size (the "D4" in `SDLM-3B-D4`) -- other checkpoints in the
SDLM family use different block sizes. Feel free to raise `max_gen_len` or lower
`threshold` for quicker (if less careful) demonstrations.

## 2.11 GraphRAG prompt template and end-to-end answer function

We combine entity linking + a chosen retrieval strategy into formatted context, wrap
it with an instruction telling the model to answer **only** from the provided
knowledge-graph facts (reducing reliance on the model's own parametric knowledge, the
whole point of RAG), and generate.


In [15]:
GRAPH_SYSTEM_PROMPT = (
    "You are a biomedical question-answering assistant. You will be given a set of "
    "facts retrieved from a knowledge graph about Alzheimer's disease, each in the form "
    "'entity --[RELATION]--> entity: evidence'. Answer the user's question using only "
    "these facts. Be concise and directly address the question."
)


def build_graph_prompt(question, edges):
    context = format_context(edges, limit=20)
    return f"Knowledge graph facts:\n{context}\n\nQuestion: {question}"


def graphrag_answer(question, method="multihop", **retrieval_kwargs):
    """method: one of '1hop', 'multihop', 'shortest_path', 'global'.
    For 'shortest_path', pass entity_a= and entity_b= in retrieval_kwargs, or omit
    them to auto-pick the first two linked entities."""
    entities = filter_dominant_hubs(link_entities(question), degree_centrality)

    if method == "1hop":
        edges = retrieve_1hop(entities, G)
    elif method == "multihop":
        edges, _ = retrieve_multihop(entities, G, degree_centrality, **retrieval_kwargs)
    elif method == "shortest_path":
        a = retrieval_kwargs.get("entity_a", entities[0] if entities else None)
        b = retrieval_kwargs.get("entity_b", entities[1] if len(entities) > 1 else None)
        if a is None or b is None:
            edges = retrieve_1hop(entities, G)  # fallback: not enough entities for a path
        else:
            sp = retrieve_shortest_path(a, b, G)
            edges = sp["edges"] if sp else retrieve_1hop(entities, G)
    elif method == "global":
        _, edges = retrieve_global_context(G, degree_centrality, **retrieval_kwargs)
    else:
        raise ValueError(f"Unknown method: {method}")

    prompt = build_graph_prompt(question, edges)
    answer = generate_with_sdlm(prompt, system_prompt=GRAPH_SYSTEM_PROMPT)
    return {"question": question, "method": method, "entities": entities, "edges": edges, "answer": answer}

## 2.12 Showcase: all four retrieval methods on real eval questions

We load the actual 30-question evaluation set and run one example per retrieval
method, printing the linked entities, retrieved context, and generated answer
alongside the gold `expected_answer` for comparison.


In [16]:
eval_data = json.load(open(EVAL_PATH, encoding="utf-8"))
questions_by_id = {q["question_id"]: q for q in eval_data["questions"]}


def show_result(result, gold_answer):
    print(f"Method: {result['method']}")
    print(f"Question: {result['question']}")
    print(f"Linked entities: {result['entities']}")
    print(f"\nRetrieved context ({len(result['edges'])} edges):")
    print(format_context(result["edges"], limit=8))
    print(f"\nGenerated answer:\n{result['answer']}")
    print(f"\nGold expected answer:\n{gold_answer}")
    print("=" * 100)


In [17]:
# Method 1: 1-hop -- simple, single-entity fact lookup (q07: memantine's mechanism)
q = questions_by_id["q07"]
result = graphrag_answer(q["question"], method="1hop")
show_result(result, q["expected_answer"])


Method: 1hop
Question: What is memantine's mechanism of action in treating Alzheimer's disease?
Linked entities: ['Memantine']

Retrieved context (2 edges):
- Memantine --[ANTAGONIST_OF]--> NMDA Receptor: Memantine is an uncompetitive NMDA receptor antagonist. (source: pubmed_12672860;wikipedia_memantine)
- Memantine --[TREATS]--> Alzheimer's Disease: A placebo-controlled trial found memantine reduced clinical deterioration in moderate-to-severe AD. (source: pubmed_12672860)

Generated answer:
Memantine's mechanism of action in treating Alzheimer's disease is as an uncompetitive NMDA Receptor antagonist, acting as an uncompetitive antagonist.

Gold expected answer:
Memantine is an uncompetitive NMDA (N-methyl-D-aspartate) receptor antagonist that reduces glutamate-driven excitotoxicity.


In [18]:
# Method 2: hub-avoiding multi-hop -- chained reasoning (q12: sleep -> glymphatic -> aggregation)
q = questions_by_id["q12"]
result = graphrag_answer(q["question"], method="multihop", max_hops=2)
show_result(result, q["expected_answer"])


Method: multihop
Question: Trace the chain of events by which poor sleep is thought to contribute to dementia-related protein aggregation.
Linked entities: ['SleepDisturbance', 'Dementia']

Retrieved context (42 edges):
- Sleep Disturbance --[PRECEDES_ONSET_OF]--> Dementia: Disrupted sleep architecture frequently precedes clinical dementia onset. (source: pubmed_33004510)
- Physical Activity --[PROTECTIVE_FACTOR_FOR]--> Dementia: Physical activity is also associated with lower all-cause and vascular dementia incidence. (source: pubmed_35301183)
- Alzheimer's Disease --[SUBTYPE_OF]--> Dementia: AD accounts for roughly 60-70% of all dementia cases worldwide. (source: wikipedia_dementia)
- Vascular Dementia --[SUBTYPE_OF]--> Dementia: Vascular dementia is the second most common dementia subtype. (source: wikipedia_dementia)
- Sleep Disturbance --[IMPAIRS]--> Glymphatic System: Poor sleep reduces the glymphatic system's brain-clearance activity, which is greatest during sleep. (source: pub

In [19]:
# Method 3: shortest path -- explicit reasoning chain between two entities (q14: Down syndrome -> amyloid)
q = questions_by_id["q14"]
result = graphrag_answer(q["question"], method="shortest_path", entity_a="DownSyndrome", entity_b="AmyloidBeta")
show_result(result, q["expected_answer"])


Method: shortest_path
Question: Why is Down syndrome considered a natural model supporting the amyloid cascade hypothesis, tracing the full genetic-to-molecular chain?
Linked entities: ['AmyloidCascadeHypothesis', 'DownSyndrome']

Retrieved context (3 edges):
- Down Syndrome --[EXTRA_COPY_OF]--> Chromosome 21: Trisomy 21 gives an extra copy of chromosome 21 and therefore an extra APP gene copy. (source: textbook_statpearls_alzheimer_disease)
- Amyloid Precursor Protein / APP gene --[LOCATED_ON]--> Chromosome 21: The APP gene resides on chromosome 21. (source: textbook_statpearls_alzheimer_disease)
- Amyloid Precursor Protein / APP gene --[CLEAVED_TO_PRODUCE]--> Amyloid-beta (Abeta): Proteolytic cleavage of APP produces the amyloid-beta fragment central to AD pathology. (source: pubmed_31753135)

Generated answer:
Down syndrome is considered natural model supporting the amyloid cascade hypothesis because it is a model supporting model supporting the full genetic-to-molecular chain.

Gol

In [20]:
# Method 4: centrality-guided global context -- broad synthesis question (q23: genetics + lifestyle synthesis)
q = questions_by_id["q23"]
result = graphrag_answer(q["question"], method="global", top_k=12)
show_result(result, q["expected_answer"])


Method: global
Question: Synthesize how genetic factors (APOE, PSEN1/2, APP) and modifiable lifestyle/environmental factors together shape an individual's overall Alzheimer's risk profile.
Linked entities: ['PSEN1', 'APOE', 'APP']

Retrieved context (16 edges):
- Alzheimer's Disease --[SUBTYPE_OF]--> Dementia: AD accounts for roughly 60-70% of all dementia cases worldwide. (source: wikipedia_dementia)
- Vascular Dementia --[COMORBID_WITH]--> Alzheimer's Disease: AD is frequently diagnosed alongside vascular dementia as mixed dementia. (source: wikipedia_dementia)
- Vascular Dementia --[SUBTYPE_OF]--> Dementia: Vascular dementia is the second most common dementia subtype. (source: wikipedia_dementia)
- Mild Cognitive Impairment --[PRODROMAL_STAGE_OF]--> Alzheimer's Disease: Amnestic MCI is frequently a prodromal stage that converts to AD at roughly 10-15% per year. (source: wikipedia_mild_cognitive_impairment)
- Amyloid-beta (Abeta) --[AGGREGATES_INTO]--> Amyloid Plaques: Amyloid-beta o

## 2.13 Comparing the four retrieval strategies

| Method | Best suited for | Weakness |
|---|---|---|
| **1-hop neighbor** | Single-entity fact lookup ("what is X's mechanism?") | Misses anything requiring 2+ connected facts |
| **Multi-hop (hub-avoiding BFS)** | Chained reasoning through non-hub intermediate concepts | Chain quality depends on the hub threshold; too low = still explodes, too high = misses real chains through moderately-connected nodes |
| **Shortest path** | Explicit "how does A relate to B" questions with 2+ named entities | Needs at least two successfully linked entities; picks *one* path even when multiple equally-valid paths exist |
| **Centrality-guided global** | Broad synthesis questions with no single clear entity anchor | Not targeted -- may include hub-adjacent facts irrelevant to the specific question |

None of these four strategies is strictly best -- this is exactly why Notebook 4
combines GraphRAG with VectorRAG rather than picking one retrieval style. A production
GraphRAG system would typically **route** a question to the right strategy (or combine
several) based on how many entities it can confidently link and how the question is
phrased, rather than committing to one method for every query.

**Next:** [`03_vectorrag_pipeline.ipynb`](03_vectorrag_pipeline.ipynb) builds the
complementary VectorRAG pipeline over the chunked article text, then
[`04_hybridrag_and_evaluation.ipynb`](04_hybridrag_and_evaluation.ipynb) combines both
and scores all three approaches on the full 30-question evaluation set.
